# பாடம் 18: குறியாக்கத்துடன் AI முகவரிகளை பாதுகாக்குதல்

## நடைமுறை நோட்புக்

இந்த நோட்புக் நான்கு பயிற்சிகள் வழியாக செல்கிறது:

1. முகவர் கருவி அழைப்புக்கு உங்கள் முதல் ரசீதை கையொப்பமிடுங்கள் மற்றும் அதை சரிபார்க்கவும்.
2. ரசீதை மாற்றி சீட்டிங் தோல்வியடைவதை கண்டு பழகுங்கள்.
3. மூன்று ரசீதை சங்கிலியை கட்டியமைத்து சங்கிலி ஒருங்கிணைப்பை உறுதிசெய்யுங்கள்.
4. மைக்ரோசாஃப்ட் முகவர் கட்டமைப்பு கருவி அழைப்பை வைத்து ஒவ்வொரு செயலுக்கும் ரசீதை உருவாக்கவும்.

அனைத்து குறியாக்க அடிப்படைக் கூறுகளும் நன்றாக பராமரிக்கப்படும் நூலகங்களிலிருந்து இறக்குமதி செய்யப்படுகின்றன (`pynacl` Ed25519க்காக, `jcs` RFC 8785 canonical JSONக்காக, Python मानक நூலகத்தில் இருந்து SHA-256க்காக `hashlib`). ரசீதுக்கான புரிதல் என்பது படிக்கவும் திருத்தவும் முடிவூண்ட Python கொடியாகும்.

செல் தொகுதிகள் வரிசையாக இயக்கவும். ஒவ்வொரு பிரிவும் குறுகியதும் சுயவசப்பட்டதும் ஆகும்.


## அமைப்பு

இரு சார்புகளையும் நிறுவவும். இரண்டிலும் அனுமதி வழங்கும் உரிமங்கள் (அப்பாச்சி-2.0 / MIT) உள்ளன.


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## உதவி பயன்பாடுகள்

இந்த இரண்டு உதவிகள் base64url குறியாக்கம் (pad செய்யாமல்) மற்றும் எந்தவொரு பொருளின் SHA-256 ஹாஷிங்கை கையாள்கின்றன. அவை நோட்புக் மீதியுள்ள பொறுப்புக்களை ரிசீட் லாஜிக்கில் மட்டுமே கவனம் செலுத்த வைக்கின்றன.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## பிரிவு 1: உங்கள் முதல் ரசீதை கையொப்பமிடுங்கள்

**Contoso Travel** நிறுவனத்தின் எஜெண்ட் தொடர்ச்சியாக சிட்னியிலிருந்து லாஸ் அஞ்சலெசுக்கு விமானங்களை ஒரு வாடிக்கையாளருக்காகத் தேடியவர் என்று நினைத்துக்கொள்ளுங்கள். நாங்கள் இந்த கருவி அழைப்பை ஒரு கையொப்பமிடப்பட்ட ரசீதாக பதிவு செய்ய விரும்புகிறோம், அதனால் ஒரு எதிர்கால ஆய்வாளர் எங்களை நம்பாமல் அதை சரிபார்க்க முடியும்.

### படி 1.1: கையொப்பமிடும் விசையை உருவாக்குக

தயாரிப்பில், எஜெண்டின் கையொப்ப விசை ஒரு ஹார்ட்வேர் பாதுகாப்பு மொட்யூல் (HSM), அசுரு கீ வால்ட் அல்லது அதேபோன்ற பாதுகாக்கப்பட்ட சேமிப்பில் இருக்கும். இந்த பாடத்திற்கு நாங்கள் நினைவகத்தில் ஒரு புதிய விசையை உருவாக்குகிறோம்.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### படி 1.2: ரசீது பेलோடை கட்டமைக்கவும்

பेलோடு ரசீதால் என்ன உறுதிப்படுத்தப்பட வேண்டும் என்பதை அனைத்தும் கொண்டுள்ளது: யார் செயல்படுத்தினார், எந்த கருவி, எந்த ஆவணங்களுடன், என்னத் திரும்பியது, எந்த கொள்கையின் கீழ், மற்றும் எப்போது. args மற்றும் முடிவுகளை நேரடியாக சேர்க்காமல் அவற்றின் ஹேஷ் செய்தலின் மூலம் செய்கிறோம், அதனால் ரசீது உணர்ச்சிமிக்க உள்ளடக்கத்தை தாக்குவதைத் தடுப்பதாகும்.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### படி 1.3: ரசீதினை கையொப்பமிட்டு ஒன்று சேர்க்கவும்

மூன்று படிகள்:

1. இரண்டு செயல்பாடுகளும் ஒரே தர்க்கரீதியான ரசீதினை உருவாக்கும் வகையில் JCS பயன்படுத்தி ப.Payloadஐ நியாயரீதியாக மாற்றுக.
2. SHA-256 உடன் நியாயப்படுத்தப்பட்ட பைட்ஸ் மீது ஹேஷ் செய்யவும்.
3. Ed25519 தனிப்பட்ட விசையுடன் ஹேஷை கையொப்பமிடவும்.

கையொப்பம் அப்படியே அதன் ஆரம்ப Payload உடன் இணைக்கப்பட்டு இறுதி ரசீதினை உருவாக்குகிறது.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### படி 1.4: ரசீதைச் சரிபார்க்கவும்

சரிபார்ப்பு செயல்முறையை எதிர்மறையாகச் செயல்படுத்தும். நாங்கள் கையொப்பத்தை அகற்றி, canonical ஹாஷைப் புதிதாக கணக்கிட்டு, ரசீதிலுள்ள பொது விசையுடன் கையொப்பத்தை ஒப்புக் கொள்கிறோம்.

இந்த சரிபார்ப்பைச் செய்யும் கணக்காய்வாளர் எங்களிடமிருந்து வேறு எதையும் வேண்டாம், மேலும் ரசீதையே தேவையாக இருக்கிறது. எந்த சேவையையும் அழைக்க வேண்டியதும், விசை அடைவகத்தில் விசார் படுத்த வேண்டியதும், நம்பிக்கை கொண்டிருக்க வேண்டியதும் இல்லை.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

நீங்கள் `Receipt is valid: True` என்று காண வேண்டும். முகவர் அதன் முதல் குறியாக்க முறையில் ஒப்புதல் பெற்ற கணக்கு பதிவை உருவாக்கியுள்ளது.


## பிரிவு 2: ரசீதை மாற்றுதல்

ரசீது என்றாலே அது தட்டச்சு மாற்றத்தை வெளிப்படுத்தவேண்டும். அதைக் கூனப்படுத்தி காண்போம்.

ரசீதிலிருந்து ஒரு எழுத்தை மட்டும் மாற்றி சரிபார்ப்பில் தோல்வியடையுமோ என பார்க்கிறோம்.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### என்ன刚刚 நடந்தது?

நாம் `policy_id` ஐ மாற்றினபோது, canonical bytes மாறின. அந்த திருப்பங்களின் SHA-256 ஹாஷ் மாறின. ஒப்புதல் (அது அசல் ஹாஷ் மீது இருந்தது) புதிய ஹாஷுடன் பொருந்தவில்லை. சான்றிதழ் சரியாக `False` ஐ திருப்புகிறது.

ரசீது எந்தப் பகுதியையும் மாற்றி அதை சரிபார்க்கச் செய்ய வழியில்லை, கருதுகவன் தனியுரிமை விசையை கொண்டிருக்கவில்லை என்றால். தனியுரிமை விசை ஓர் விசை அரணில் இருந்தால் மற்றும் பொதுவிசை வெளியிடப்பட்டு இருந்தால், மாற்றத்தை மறைக்க முடியாது.

உங்களுக்குப் பிரயோகம் செய்துபாருங்கள்: மேலே உள்ள செலில் `tool_name` அல்லது `agent_id` அல்லது `timestamp` ஐ மாற்றி மீண்டும் நடத்துங்கள். ஒவ்வொரு மாற்றத்தும் செல்லாத ரசீதத்தை உருவாக்கும்.


## பகுதி 3: ரசீதுகளை தொடர் இணைத்தல்

ஒரு ஒற்றை ரசீது ஒரு செயலைக் காக்கும். பெரும்பாலான முகவரிகள் பல செயல்களை மேற்கொள்கின்றனர். முழு தொடரையும் மாற்றமுள்ளதாக்காமல் செய்ய, ஒவ்வொரு ரசீதையும் முன் ரசீதின் ஹாஷ் புதிய ரசீதின் பணியில் சேர்ப்பதன் மூலம் முன் ரசீதிற்கு இணைக்கிறோம்.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

யாராவது ரசீதுகளை நீக்கினால் அல்லது மறுசீரமைத்தால், தொடர் அதே இடத்தில் முறைகேடாகிறது. பிறகு வரும் எந்த ரசீதின் உறுதிப்படுத்தலும் its `previous_receipt_hash` அதன் முன்னோடியின் உண்மையான ஹாஷுடன் பொருந்தாததால் தோல்வியடைகிறது.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

இப்போது சரங்களின் நடுப்பகுதியை திருத்தி சங்கிலியை துண்டிக்கவும் மற்றும் மறுபரிசோதனை செய்யவும். திருத்தப்பட்ட ரசீதை அதன் கையொப்ப பரிசோதனையில் தோல்வி அடைகிறது, மேலும் அடுத்த ரசீதை அதன் சங்கிலி-இணைப்பு பரிசோதனையில் தோல்வியடைகிறது (ஏனெனில் அதன் `previous_receipt_hash` மாற்றப்பட்ட நடுப்பகுதி ரசீதை ஹாஷுடன் இனி பொருந்தவில்லை).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

ரசீத்ய் 0 இன்னும் சரிபார்க்கிறது (இதில் மாற்றம் செய்யப்படவில்லை மற்றும் சார்ந்த முன்கூறு இல்லை). ரசீத்ய் 1 அதன் கையொப்பச் சோதனையில் தோல்வி அடைகிறது ஏனெனில் நாங்கள் `tool_args_hash` ஐ மாற்றினோம். ரசீத்ய் 2 அதன் சங்கிலி-இணை சோதனையில் தோல்வி அடைகிறது ஏனெனில் அதன் `previous_receipt_hash` அசல் (இப்போது மாற்றப்பட்ட) ரசீத்ய் 1 மீது கணக்கிடப்பட்டது.

மாற்றிய ரசீத்ய் 1 ஐ ஒரு தாக்குதல் நடாதவர் மீண்டும் கையொப்பமிடினாலும் (அவர்கள் தனியுரிமைக் குஞ்சியின்றி முடியாது), ரசீத்ய் 2 இல் சங்கிலி-இணையின் பொருத்தம் இல்லாதிருப்பது மாற்றத்தை வெளிப்படுத்தும். மாற்றத்தை மறைக்க, தாக்குதல் நடாதவர் மாற்றம் செய்யப்பட்ட இடத்திலிருந்து அனைத்து ரசீத்ய்களையும் மீண்டும் கையொப்பமிட வேண்டியிருக்கும், அதற்கு தனியுரிமைக் குஞ்சியின் உடைமை அவசியம்.


## பிரிவு 4: ஒரு ஏஜென்ட் கருவி அழைப்புக்கு ரசீது கையெழுத்து இணைத்து கட்டுரைச் செய்யவும்

உண்மையான செயலாக்கத்தில், ஒவ்வொரு ஏஜென்ட் ஆசிரியரும் `make_receipt` ஐ அழைக்க நினைவிருக்க வேண்டும் என்பதில்லை. ஒவ்வொரு கருவி அழைப்புக்கும் ரசீது கையெழுத்து தானாகவே ஆக வேண்டும்.

இதோ மிகவும் எளிய மாதிரி: ஏதேனும் அழைக்கக்கூடிய கருவி செயல்பாட்டைப் பெற்று அதற்காக ரசீது வெளியிடும் பதிப்பை திருப்பி வழங்கும் ஒரு மூடுபணி வகுப்பு. இது Microsoft Agent Framework (`agent_framework.foundry`) உட்பட எந்த ஏஜென்ட் கட்டமைப்புக்கும் பொருந்தும்.

நீங்கள் Microsoft Foundry திட்டத்தை அமைக்கவில்லை என்றாலும், கீழே உள்ள உள்ளூர் மோக் மாதிரியை எடுத்துக்காட்டுகிறது.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


### Microsoft Agent Framework உடன் ஒருங்கிணைப்பு

மேல் உள்ள `ReceiptedTool` பொருத்தி (wrapper) கட்டமைப்பு-தனிமை இல்லாதது. அதனை Microsoft Agent Framework உடன் உருவாக்கப்பட்ட ஏஜென்டில் பயன்படுத்த, உடைக்கப்பட்ட செயல்பாட்டை ஒரு கருவியாக பதிவு செய்யவும். ஒரு வரைபடம் (நீங்கள் mock ஐ நிஜ Microsoft Foundry கருவி பதிவு மூலம் மாற்றுவீர்கள்):

```python
# ஒருங்கிணைப்பு வடிவத்தை காட்டும் போலி குறியீடு.
# os ஐ இறக்குமதி செய்க
# agent_framework.foundry இலிருந்து FoundryChatClient ஐ இறக்குமதி செய்க
# azure.identity இலிருந்து AzureCliCredential ஐ இறக்குமதி செய்க
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="நீங்கள் ஒரு Contoso பயண முகவர் ...",
#     tools=[receipted_lookup],   # மூடிய கருவி, அசல் செயல்பாடு அல்ல
# )
# response = agent.run("சிட்னியிலிருந்து லாஸ் ஆஞ்சலஸுக்கு ஜூன் மாதத்தில் ஓட்டுப்படிகள் காண்க.")
#
# # இயக்கம் முடிந்த பிறகு, முகவர் பயன்படுத்திய ஒவ்வொரு கருவிக்கும் முன்பு கையொப்பமிடப்பட்ட ரசீதை உள்ளது:
# audit_chain = receipted_lookup.receipts
```

ஏஜென்ட் கட்டமைப்புக்கு ரசீதைப் பற்றி எந்த விளக்கம் தேவையில்லை. ரசீதை கையெழுத்திடல் கருவியின் சுற்றிலும் பொருத்தப்பட்டுள்ளது, கட்டமைப்பில் செதுக்கப்படவில்லை. இது ஏஜென்டை மறுபடிக்காமல், இருந்து வரும் ஏஜென்ட் குறியீடுக்கு மாறுகிறத்தையும் (provenance) சேர்க்கும் முறையாகும்.


## மறுபரிசீலனை மற்றும் விரிவான சவால்

நீங்கள் செய்துள்ளீர்கள்:

- Ed25519 முக்கிய ஜோடியை உருவாக்கியுள்ளீர்கள்.
- முகவர் கருவி அழைப்பிற்கான ரசீதை உருவாக்கி கையொப்பமிட்டுள்ளீர்கள்.
- சர்வதேச திறவுகோல் கொண்டு ரசீதையை ஆன்லைன் தவிர புதுப்பித்து சரிபார்த்துள்ளீர்கள்.
- ஒரு ரசீதையை மாற்றி அவ்வாறு சரிபார்த்ததும் தோல்வியடைந்ததை பார்த்துள்ளீர்கள்.
- மூன்று ரசீதைகளின் ஹேஷ்-சேன்செய்த வரிசையை உருவாக்கியுள்ளீர்கள்.
- சங்கிலி நடுவில் தற்காலிக மாற்றம் செய்து கையொப்பத் தோல்வியும் சங்கிலி-இணை தோல்வியும் காணப்பட்டன.
- ஒரு கருவி செயல்பாட்டை தானாக ரசீதை கையொப்பம் செய்வதற்காக சார்ந்துள்ளீர்கள்.

**விரிவான சவால்.** ரசீதின் அமைப்பில் `request_id` புலத்தை (பங்கு பெறும் தடயத்திற்கு UUID) சேர்த்துக் கொள்ளுங்கள். அதற்கு `make_receipt` ஐ புதுப்பித்து, ரசீதைகள் முடிவட்டமாக சரிபார்க்கப்படுவதைக் உறுதிப்படுத்துங்கள். பின்னர் கையொப்பம் முடிந்ததற்குப் பிறகு புலத்தை மாற்றி சரிபார்ப்பு தோல்வியில் முடிவடைவதை பதிப்புரிமையிடுங்கள். இது ஒவ்வொரு கானோனிக்கல் குறியீடு பைட்டும் கையொப்பத்தில் எப்படி பங்குபெறுவதாக உள்ளது என்பதை நீங்கள் எடுத்துக்கொள்ள வைக்கிறது.

**முக்கிய எல்லை.** ரசீதைகள் மூன்று விஷயங்களை மட்டும் நிரூபிக்கின்றன: உரிமை (இந்த திறவுகோல் இந்த உள்ளடக்கத்தில் கையொப்பமிட்டது), ஒருங்கிணைப்பு (கையொப்பம் பிறகு உள்ளடக்கம் மாற்றம் செய்யப்படவில்லை), மற்றும் வரிசை (இந்த ரசீதையானது அந்த ரசீதைக்கு பிறகு வந்தது). அவை முகவரின் செயல்பாடு சரியானது என்பதை, `policy_id` இல் பெயர் சொல்லிய கொள்கை பின்பற்றப்பட்டது என்பதை, அல்லது முகவர் ஒவ்வொரு விதியையும் பின்பற்றினான் என்பதையும் நிரூபிக்காது. ரசீதைகள் ஒரு அடித்தளமாகின்றன. நிர்வாகம் அதில் மேல் நீங்கள் கட்டிய அமைப்பு ஆகும்.

அந்த எல்லையை மனதில் வைத்து பாடம் README ஐ மீண்டும் படியுங்கள். ரசீதையுடன் அணிமுகம் ஏற்றுக்கொள்ளும் பொதுவான தவறான கருத்து "மாம் ரசீதை இருக்கிறது" என்றால் "நாம் ஆட்சி செய்யப்படுகிறோம்" என்பதுதான். அது அல்ல. ரசீதை முகவர் செயல்பாடுகளை கணக்கிட முடியும். அவை அதை சரியாகக் கொண்டுவராது.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
